# 🌱 AgriPulse — Reproducible Proof of Concept

**Explainable Crop Stress & Inspection Intelligence**  
Precision Agriculture & Crop Intelligence • Team T0034

This notebook reproduces the **decision/output layer** of the submitted AgriPulse PoC from the committed hotspot dataset. It loads the provided hotspot CSV and summary JSON, validates the outputs, ranks inspection priorities, and generates a geographic visualization.

> **Scope note:** the current submitted PoC package contains the processed hotspot outputs used by the Streamlit dashboard. This notebook does not claim to recreate unavailable raw Sentinel-2 imagery processing; it makes the submitted output reproducible and inspectable from the included files.

## 1. Setup

The notebook uses the same lightweight Python stack as the dashboard: Pandas for tabular analysis and Folium for the geographic output.

In [ ]:
from pathlib import Path
import json
import pandas as pd

try:
    import folium
except ImportError as exc:
    raise ImportError(
        "Folium is required. Install project dependencies with: pip install -r requirements.txt"
    ) from exc

PROJECT_ROOT = Path.cwd()
CSV_PATH = PROJECT_ROOT / "agri_pulse_hotspots_final.csv"
SUMMARY_PATH = PROJECT_ROOT / "agri_pulse_summary.json"

assert CSV_PATH.exists(), f"Missing input file: {CSV_PATH}"
assert SUMMARY_PATH.exists(), f"Missing summary file: {SUMMARY_PATH}"

print(f"Project root: {PROJECT_ROOT.resolve()}")
print("Input files found: OK")

## 2. Load the submitted PoC outputs

In [ ]:
hotspots = pd.read_csv(CSV_PATH)
with open(SUMMARY_PATH, "r", encoding="utf-8") as f:
    summary = json.load(f)

print(f"Hotspot rows: {len(hotspots):,}")
print("Columns:", list(hotspots.columns))
display(hotspots.head())

## 3. Validate the hotspot output schema

The dashboard and pitch deck describe each hotspot using ranking/priority information, area, stress score, and geographic coordinates. Column matching below is deliberately tolerant of common naming variations.

In [ ]:
def find_column(df, candidates):
    normalized = {str(c).strip().lower().replace(" ", "_"): c for c in df.columns}
    for candidate in candidates:
        key = candidate.strip().lower().replace(" ", "_")
        if key in normalized:
            return normalized[key]
    return None

rank_col = find_column(hotspots, ["rank", "priority_rank", "hotspot_rank"])
priority_col = find_column(hotspots, ["priority", "priority_level"])
area_col = find_column(hotspots, ["area_ha", "area", "hotspot_area_ha"])
score_col = find_column(hotspots, ["stress_score", "score", "stress"])
lat_col = find_column(hotspots, ["latitude", "lat"])
lon_col = find_column(hotspots, ["longitude", "lon", "lng"])

required = {
    "area": area_col,
    "stress score": score_col,
    "latitude": lat_col,
    "longitude": lon_col,
}
missing = [name for name, col in required.items() if col is None]
assert not missing, f"Required output columns not found: {missing}. Found: {list(hotspots.columns)}"

print("Required geographic/priority fields: OK")
print({"rank": rank_col, "priority": priority_col, "area": area_col, "stress_score": score_col, "latitude": lat_col, "longitude": lon_col})

## 4. Reproduce the headline PoC metrics

The submitted README reports **10,473 persistent-stress pixels, 1,048 high-priority pixels, 35 actionable hotspots, and 9.16 ha total hotspot area**. The hotspot count and total area can be independently checked from the included CSV.

In [ ]:
headline = {
    "Persistent Stress pixels": summary.get("persistent_stress_pixels", summary.get("persistent_stress")),
    "High Priority pixels": summary.get("high_priority_pixels", summary.get("high_priority")),
    "Actionable Hotspots": len(hotspots),
    "Hotspot Area (ha)": round(pd.to_numeric(hotspots[area_col], errors="coerce").sum(), 2),
}

display(pd.DataFrame([headline]))
assert len(hotspots) == 35, f"Expected 35 hotspots from the submitted PoC, found {len(hotspots)}"
assert headline["Hotspot Area (ha)"] == 9.16, f"Expected 9.16 ha, found {headline['Hotspot Area (ha)']}"
print("Headline hotspot outputs validated: OK")

## 5. Rank inspection priorities

AgriPulse is designed to turn a large spatial signal into a manageable inspection list. The table below surfaces the highest-ranked hotspots available in the submitted output.

In [ ]:
if rank_col is not None:
    ranked = hotspots.sort_values(rank_col, ascending=True)
else:
    ranked = hotspots.sort_values(score_col, ascending=False)

display(ranked.head(10))

## 6. Geographic inspection map

The same hotspot coordinates used by the dashboard are rendered as an interactive Folium map. This is the final decision-support output: **where should the field team inspect first?**

In [ ]:
lat = pd.to_numeric(hotspots[lat_col], errors="coerce")
lon = pd.to_numeric(hotspots[lon_col], errors="coerce")
valid = hotspots.loc[lat.notna() & lon.notna()].copy()

assert len(valid) > 0, "No valid geographic coordinates found in the hotspot output."

center = [pd.to_numeric(valid[lat_col]).mean(), pd.to_numeric(valid[lon_col]).mean()]
m = folium.Map(location=center, zoom_start=11, control_scale=True)

for _, row in valid.iterrows():
    label_parts = []
    if rank_col is not None:
        label_parts.append(f"Rank: {row[rank_col]}")
    if priority_col is not None:
        label_parts.append(f"Priority: {row[priority_col]}")
    label_parts.append(f"Area: {row[area_col]} ha")
    label_parts.append(f"Stress score: {row[score_col]}")
    popup = "<br>".join(label_parts)
    folium.CircleMarker(
        location=[row[lat_col], row[lon_col]],
        radius=6,
        popup=popup,
        fill=True,
    ).add_to(m)

m

## 7. Reproducibility check

The notebook finishes with explicit checks so a reviewer can run it from the repository and verify that the submitted hotspot output is intact.

In [ ]:
assert CSV_PATH.exists()
assert SUMMARY_PATH.exists()
assert len(hotspots) == 35
assert round(pd.to_numeric(hotspots[area_col], errors="coerce").sum(), 2) == 9.16
assert pd.to_numeric(hotspots[lat_col], errors="coerce").notna().all()
assert pd.to_numeric(hotspots[lon_col], errors="coerce").notna().all()

print("✅ AgriPulse notebook completed successfully.")
print("✅ Input files loaded")
print("✅ Hotspot schema validated")
print("✅ 35 hotspots reproduced")
print("✅ 9.16 ha hotspot area reproduced")
print("✅ Interactive geographic output generated")